In [ ]:
#!/usr/bin/env python3
import numpy as np
import matplotlib.pyplot as plt
import json

# ----------------- Simplicial structure -----------------
N_NODES = 10

EDGES = [
    (0, 1), (0, 2), (0, 3), (1, 2), (2, 3), (2, 8), (2, 9),
    (3, 4), (3, 5), (4, 5),
    (5, 6), (5, 8), (5, 9),
    (6, 7), (6, 8), (7, 8), (7, 9), (8, 9),
]

TRIANGLES = [
    (0, 1, 2),
    (7, 8, 9),
    (3, 4, 5),
    (6, 7, 8),
    (2, 8, 9),
]

# ----------------- Build adjacency structures -----------------
edge_neighbors = [[] for _ in range(N_NODES)]
for i, j in EDGES:
    edge_neighbors[i].append(j)
    edge_neighbors[j].append(i)

triangles_by_node = [[] for _ in range(N_NODES)]
for a, b, c in TRIANGLES:
    triangles_by_node[a].append((b, c))
    triangles_by_node[b].append((a, c))
    triangles_by_node[c].append((a, b))

# ----------------- Kuramoto + simplicial dynamics -----------------
def kuramoto_simplicial(theta, omega, K1, K2):
    """
    Standard Summation Form (No Mean-Field Normalization).
    """
    N = len(theta)
    dtheta = np.zeros_like(theta)
    pair_term = np.zeros_like(theta)
    tri_term = np.zeros_like(theta)

    for i in range(N):
        # Pairwise edges
        if edge_neighbors[i]:
            sum_edges = 0.0
            for j in edge_neighbors[i]:
                sum_edges += np.sin(theta[j] - theta[i])
        else:
            sum_edges = 0.0

        # Triangles
        if triangles_by_node[i]:
            sum_tris = 0.0
            for (j, k) in triangles_by_node[i]:
                sum_tris += np.sin(theta[j] + theta[k] - 2.0 * theta[i])
        else:
            sum_tris = 0.0

        pair_term[i] = K1 * sum_edges
        tri_term[i]  = K2 * sum_tris
        dtheta[i]    = omega[i] + pair_term[i] + tri_term[i]

    return dtheta, pair_term, tri_term

# ----------------- Simulation -----------------
def simulate_kuramoto_simplicial(
    T_total=30.0,
    dt=1/256,
    K1=5.5,
    K2=9.5,
    freq_spread=15.0,
    seed=123,
):
    np.random.seed(seed)
    steps = int(T_total / dt)
    t = np.linspace(0.0, T_total, steps)

    # High variance natural frequencies
    omega = np.random.uniform(-freq_spread, freq_spread, size=N_NODES)
    theta = np.random.uniform(0.0, 2.0 * np.pi, size=N_NODES)

    theta_traj = np.zeros((steps, N_NODES))
    pair_traj  = np.zeros((steps, N_NODES))
    tri_traj   = np.zeros((steps, N_NODES))

    for s in range(steps):
        theta_traj[s] = theta

        k1, pair1, tri1 = kuramoto_simplicial(theta, omega, K1, K2)
        pair_traj[s] = pair1
        tri_traj[s]  = tri1

        k2, _, _ = kuramoto_simplicial(theta + 0.5 * dt * k1, omega, K1, K2)
        k3, _, _ = kuramoto_simplicial(theta + 0.5 * dt * k2, omega, K1, K2)
        k4, _, _ = kuramoto_simplicial(theta + dt * k3,        omega, K1, K2)

        theta = theta + (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)

    return t, theta_traj, pair_traj, tri_traj, omega

# ----------------- Run + Save -----------------
if __name__ == "__main__":

    T_TOTAL     = 30.0
    DT          = 1/256
    K1          = 5.5
    K2          = 9.5
    FREQ_SPREAD = 19.9
    SEED        = 42

    t, theta_traj, pair_traj, tri_traj, omega = simulate_kuramoto_simplicial(
        T_total=T_TOTAL,
        dt=DT,
        K1=K1,
        K2=K2,
        freq_spread=FREQ_SPREAD,
        seed=SEED,
    )

    X = np.sin(theta_traj)
    dtheta_traj = pair_traj + tri_traj + omega[np.newaxis, :]

    np.savetxt("kuramoto_simplicial_X.csv",      X,            delimiter=",")
    np.savetxt("kuramoto_simplicial_dtheta.csv", dtheta_traj,  delimiter=",")
    np.savetxt("kuramoto_simplicial_theta.csv",  theta_traj,   delimiter=",")

    print(f"Data Generated: T={T_TOTAL}s, Fs={1/DT}Hz")

    # JSON Metadata
    edge_coeff = {}
    for i, j in EDGES:
        edge_coeff[f"{i}-{j}"] = K1
        edge_coeff[f"{j}-{i}"] = K1

    tri_coeff = {}
    for (a, b, c) in TRIANGLES:
        for i, (j, k) in [(a, (b, c)), (b, (a, c)), (c, (a, b))]:
            key_i = str(i)
            if key_i not in tri_coeff: tri_coeff[key_i] = {}
            tri_coeff[key_i][f"{j}-{k}"] = K2

    meta = {
        "parameters": {"edge_coeff": edge_coeff, "tri_coeff": tri_coeff},
        "simulation": {
            "fs": 1.0/DT,
            "T_total": T_TOTAL,
            "K1": K1,
            "K2": K2,
            "freq_spread": FREQ_SPREAD
        }
    }
    with open("kuramoto_simplicial_meta.json", "w") as f:
        json.dump(meta, f, indent=2)

    # --- Diagnostics ---
    order_param = np.abs(np.mean(np.exp(1j * theta_traj), axis=1))
    mean_R = np.mean(order_param)
    print(f"\nMean Order Parameter R = {mean_R:.3f}")

    corr = np.corrcoef(X.T)
    off_diag = corr - np.eye(N_NODES)
    max_corr = np.max(np.abs(off_diag))
    print(f"Max Abs Off-Diagonal Correlation: {max_corr:.3f}")

    if mean_R > 0.8:
        print("\n⚠️ WARNING: System is too synchronized.")
    elif max_corr > 0.8:
        print("\n⚠️ WARNING: Correlations are high.")
    else:
        print("\n✅ System state looks excellent for SINDy identification.")

    # -------------------------------------------------------------
    # NEW SECTION: PER-NODE PERCENTAGE BREAKDOWN
    # -------------------------------------------------------------
    print("\n" + "="*80)
    print("PER-NODE DYNAMICS BREAKDOWN (Average Absolute Magnitude)")
    print("Equation: dθ/dt = ω + Pair_Term + Tri_Term")
    print("Percentages indicate how much each term drives the velocity on average.")
    print("="*80)

    header = f"{'Node':<6} | {'Natural (ω)':<16} | {'Pairwise':<16} | {'Triadic':<16} | {'Total Magnitude'}"
    print(header)
    print("-" * len(header))

    avg_pair_global = 0
    avg_tri_global = 0

    for i in range(N_NODES):
        # 1. Calculate Average Magnitude (Signal Strength) for this node
        # We use absolute values because these terms oscillate around zero.
        mag_omega = np.abs(omega[i])
        mag_pair  = np.mean(np.abs(pair_traj[:, i]))
        mag_tri   = np.mean(np.abs(tri_traj[:, i]))

        total_mag = mag_omega + mag_pair + mag_tri

        # 2. Calculate Percentages
        if total_mag > 1e-12:
            pct_omega = (mag_omega / total_mag) * 100
            pct_pair  = (mag_pair  / total_mag) * 100
            pct_tri   = (mag_tri   / total_mag) * 100
        else:
            pct_omega = pct_pair = pct_tri = 0.0

        # Accumulate for global average
        avg_pair_global += mag_pair
        avg_tri_global  += mag_tri

        # 3. Print
        # Note: 'Rest' is interpreted as Natural Frequency (omega) here.
        print(f"{i:<6} | {pct_omega:6.1f}% ({mag_omega:5.1f}) | {pct_pair:6.1f}% ({mag_pair:5.1f}) | {pct_tri:6.1f}% ({mag_tri:5.1f}) | {total_mag:6.2f}")

    print("-" * len(header))

    # Global Ratio
    avg_pair_global /= N_NODES
    avg_tri_global /= N_NODES
    ratio = avg_pair_global / avg_tri_global if avg_tri_global > 0 else 0

    print(f"Global Avg Strength -> Pairwise: {avg_pair_global:.3f}, Triadic: {avg_tri_global:.3f}")
    print(f"Global Pair/Tri Ratio: {ratio:.2f}")
    print("="*80)

In [ ]:
# Alternative: Stacked area plot to show relative contributions
fig, axes = plt.subplots(2, 5, figsize=(18, 8), sharex=True)
axes = axes.flatten()

for node in range(N_NODES):
    ax = axes[node]

    # Stack positive and negative contributions separately
    pair_pos = np.maximum(pair_traj[:, node], 0)
    tri_pos = np.maximum(tri_traj[:, node], 0)
    pair_neg = np.minimum(pair_traj[:, node], 0)
    tri_neg = np.minimum(tri_traj[:, node], 0)

    # Plot stacked areas
    ax.fill_between(t, 0, pair_pos, label='Pairwise+', alpha=0.6, color='C0')
    ax.fill_between(t, 0, tri_pos, label='Triadic+', alpha=0.6, color='C1')
    ax.fill_between(t, 0, pair_neg, alpha=0.6, color='C0', linestyle='--')
    ax.fill_between(t, 0, tri_neg, alpha=0.6, color='C1', linestyle='--')

    ax.axhline(y=0, color='black', linewidth=0.8)
    ax.set_title(f'Node {node}', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)

    if node % 5 == 0:
        ax.set_ylabel('Contribution', fontsize=9)
    if node >= 5:
        ax.set_xlabel('Time', fontsize=9)
    if node == 0:
        ax.legend(loc='upper right', fontsize=7)

plt.suptitle('Stacked Contributions (Positive/Negative) per Node',
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('kuramoto_simplicial_stacked_contributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#!/usr/bin/env python3
"""
Corrected Simplicial SINDy for your simplicial Kuramoto generator.

Key fixes vs your earlier SINDy script:
  1) NO per-node centering of theta before building sin-features (critical).
  2) Use the exact ground-truth dtheta from kuramoto_simplicial_dtheta.csv (no SG derivative).
  3) Enforce Kuramoto incidence/sign structure by design:
       - Each undirected edge (a,b) has ONE scalar weight k_e.
       - Its regressor contributes +sin(theta_b-theta_a) to node a and -sin(theta_b-theta_a) to node b.
       - Each triangle-incidence (center c | other1,other2) has ONE scalar weight k_tri.
       - Its regressor contributes only to the center node row.
  4) SOC hierarchy is applied between each triangle-incidence weight and its 3 parent edge weights:
       |k_tri| <= rho * |k_edge|  for each parent edge of that triangle.
  5) Omega is solved as unpenalized per-node intercept (10 scalars).

This script does a 2D grid search over (lambda_scale, rho_hier) and reports Edge F1 and Triangle F1.
"""

import os
import math
from itertools import combinations

import numpy as np
import pandas as pd

# ---------------- GPU optional ----------------
try:
    import cupy as cp
    GPU_AVAILABLE = True
    _ = cp.zeros(1)
    print("✓ GPU (CuPy) detected")
except Exception:
    GPU_AVAILABLE = False
    cp = None
    print("⚠️ GPU not available (using NumPy)")

xp = cp if GPU_AVAILABLE else np

# ---------------- Ground truth topology (from your generator) ----------------
N_NODES = 10
GT_EDGES = [
    (0, 1), (0, 2), (0, 3), (1, 2), (2, 3), (2, 8), (2, 9),
    (3, 4), (3, 5), (4, 5),
    (5, 6), (5, 8), (5, 9),
    (6, 7), (6, 8), (7, 8), (7, 9), (8, 9),
]
GT_TRIANGLES = [
    (0, 1, 2),
    (7, 8, 9),
    (3, 4, 5),
    (6, 7, 8),
    (2, 8, 9),
]
GT_EDGES_SET = {frozenset(e) for e in GT_EDGES}
GT_TRIS_SET  = {frozenset(t) for t in GT_TRIANGLES}

# ---------------- IO ----------------
THETA_CSV = "kuramoto_simplicial_theta.csv"
DTHETA_CSV = "kuramoto_simplicial_dtheta.csv"

OUTDIR = "sindy_kuramoto_correct"
os.makedirs(OUTDIR, exist_ok=True)

# ---------------- Hyperparams ----------------
MAX_ROWS_TIME = 8000          # cap time samples used (subsample uniformly)
ADMM_RHO = 3.0
ADMM_OVERRELAX = 1.6
MAX_ITERS = 600
TRACE_EVERY = 0              # set >0 if you want trace

ROW_NORM_NNZ_THR = 1e-8
PARAM_ABS_NNZ_THR = 1e-10

# Grid search
LAMBDA_SCALES = np.linspace(0.2, 3.0, 10)
RHO_SCALES = [0.5, 0.75, 0.90, 1.0, 1.2, 1.5]

# ---------------- Utilities ----------------
def robust_sigma(x: np.ndarray) -> float:
    """Robust sigma via MAD; x can be any shape."""
    x = np.asarray(x).ravel()
    med = np.median(x)
    mad = np.median(np.abs(x - med)) + 1e-12
    return 1.4826 * mad

def make_pairs(n: int):
    iu0, iu1 = np.triu_indices(n, k=1)
    pairs = list(zip(iu0.tolist(), iu1.tolist()))
    return pairs

def make_triples(n: int):
    return list(combinations(range(n), 3))

# ---------------- SOC projection via Dykstra (works for k=1 scalars too) ----------------
def project_rows_dykstra(Z, edges, alpha, iters=50, tol=1e-8):
    """
    Project (Z[child], Z[parent]) onto constraints ||child|| <= alpha ||parent|| for each edge constraint.
    Here Z has shape (g,k). In our use k=1 (scalar coefficients).
    """
    Zc = Z.copy()
    if not edges:
        return Zc

    ci = xp.asarray([e[0] for e in edges], dtype=xp.int32)
    pi = xp.asarray([e[1] for e in edges], dtype=xp.int32)

    E, ncols = int(ci.shape[0]), int(Zc.shape[1])
    inc_c = xp.zeros((E, ncols), dtype=Zc.dtype)
    inc_p = xp.zeros((E, ncols), dtype=Zc.dtype)

    for _ in range(iters):
        Zold = Zc.copy()
        c_tmp = Zc[ci] - inc_c
        p_tmp = Zc[pi] - inc_p

        C = xp.linalg.norm(c_tmp, axis=1)
        P = xp.linalg.norm(p_tmp, axis=1)

        feas = (C <= alpha * P + 1e-12)
        if (~feas).any():
            idx = xp.where(~feas)[0]
            Ct, Pt = C[idx], P[idx]
            cti, pti = c_tmp[idx], p_tmp[idx]

            # Projection onto { (c,p): ||c|| <= alpha ||p|| } in R^{2k}
            denom = 1.0 + alpha * alpha
            t = (alpha * Ct + Pt) / denom
            sc = xp.where(Ct > 0, (alpha * t) / Ct, 0.0)
            sp = xp.where(Pt > 0, t / Pt, 0.0)

            c_tmp[idx] = cti * sc[:, None]
            p_tmp[idx] = pti * sp[:, None]

        inc_c = c_tmp - (Zc[ci] - inc_c)
        inc_p = p_tmp - (Zc[pi] - inc_p)
        Zc[ci] = c_tmp
        Zc[pi] = p_tmp

        if float(xp.linalg.norm(Zc - Zold)) < tol:
            break

    return Zc

# ---------------- ADMM solver: group lasso (row-wise) + SOC hierarchy ----------------
def solve_admm_gl_soc(
    G, B, edges_soc, lamb, rho_hier, w,
    max_iters=500, rho_admm=3.0, overrelax=1.6,
    adaptive_rho=True, log_every=0,
    proj_every=1, proj_iters=50, proj_tol=1e-8,
):
    """
    Minimize: 0.5 ||Theta * Xi - Y||^2 + lamb * sum_i w_i ||Xi_i||_2
    subject to SOC constraints: ||Xi_child||_2 <= rho_hier ||Xi_parent||_2 for each (child,parent) in edges_soc.

    Works for k=1 (scalar coefficients) and general k.
    """
    g, k = int(G.shape[0]), int(B.shape[1])
    Xi = xp.zeros((g, k), dtype=G.dtype)
    Z  = xp.zeros_like(Xi)
    C  = xp.zeros_like(Xi)
    Uz = xp.zeros_like(Xi)
    Uc = xp.zeros_like(Xi)

    I = xp.eye(g, dtype=G.dtype)

    def factor(A):
        try:
            L = xp.linalg.cholesky(A)
            return L, True
        except Exception:
            return None, False

    rho = float(rho_admm)
    A = G + (2.0 * rho) * I
    L, chol = factor(A)

    for it in range(max_iters):
        Z_prev = Z.copy()
        C_prev = C.copy()

        # Xi-step: solve (G + 2 rho I) Xi = B + rho(Z-Uz + C-Uc)
        RHS = B + rho * (Z - Uz + C - Uc)
        if chol:
            Ytmp = xp.linalg.solve(L, RHS)
            Xi = xp.linalg.solve(L.T, Ytmp)
        else:
            Xi = xp.linalg.solve(A, RHS)

        # over-relax
        Xi_hat = overrelax * Xi + (1.0 - overrelax) * Z_prev

        # Z-step: row-wise shrinkage
        Wz = Xi_hat + Uz
        norms = xp.sqrt((Wz * Wz).sum(axis=1) + 1e-16)
        shrink = xp.maximum(0.0, 1.0 - (lamb * w) / (rho * norms))
        Z = (Wz.T * shrink).T

        # C-step: SOC projection
        if proj_every and (it % proj_every) == 0:
            C = project_rows_dykstra(Xi_hat + Uc, edges_soc, rho_hier, iters=proj_iters, tol=proj_tol)
        else:
            C = Xi_hat + Uc

        # dual updates
        Uz += (Xi_hat - Z)
        Uc += (Xi_hat - C)

        # residuals
        r_pri = float(xp.linalg.norm(Xi - Z) + xp.linalg.norm(Xi - C))
        r_dual = float(rho * xp.linalg.norm((Z - Z_prev) + (C - C_prev)))

        if log_every and (it % log_every == 0 or it == max_iters - 1):
            print(f"    it={it:4d} r_pri={r_pri:.3e} r_dual={r_dual:.3e} rho={rho:.3g}")

        # adaptive rho
        if adaptive_rho and it % 10 == 0 and it > 30:
            ratio = r_pri / max(r_dual, 1e-12)
            if ratio > 2.0:
                rho *= 1.2
                Uz /= 1.2
                Uc /= 1.2
            elif ratio < 0.5:
                rho /= 1.2
                Uz *= 1.2
                Uc *= 1.2
            rho = float(min(max(rho, 1.0), 20.0))
            A = G + (2.0 * rho) * I
            L, chol = factor(A)

        if r_pri < 1e-5 and r_dual < 1e-5:
            break

    # one last projection for safety
    Xi = project_rows_dykstra(Xi, edges_soc, rho_hier, iters=200, tol=1e-10)
    return Xi

# ---------------- Build the *correct* design matrix with incidence structure ----------------
def build_design(theta: np.ndarray, dtheta: np.ndarray, n: int, max_rows_time: int):
    """
    theta: (T,n)
    dtheta: (T,n)
    Builds a dense Theta_flat of shape (T*n, p) and Y_flat of shape (T*n,1),
    with columns:
      - omega_i intercepts: n columns (unpenalized)
      - edge weights: one column per undirected pair (a<b)
      - triangle-incidence weights: one column per (triple, center) -> 3 per triple
    """
    T = theta.shape[0]
    if T > max_rows_time:
        idx = np.linspace(0, T - 1, max_rows_time, dtype=int)
        theta = theta[idx]
        dtheta = dtheta[idx]
        T = theta.shape[0]

    pairs = make_pairs(n)
    triples = make_triples(n)

    mE = len(pairs)
    mTr = len(triples) * 3
    p = n + mE + mTr

    # Flattened target: (T*n,1)
    Y_flat = dtheta.reshape(T * n, 1).astype(np.float32)

    Theta_flat = np.zeros((T * n, p), dtype=np.float32)

    # omega intercept columns: column i is 1 on rows corresponding to node i
    t_idx = np.arange(T, dtype=int)
    for i in range(n):
        rows_i = t_idx * n + i
        Theta_flat[rows_i, i] = 1.0

    # edge columns: for each undirected edge (a<b) one scalar weight
    # contribution to node a is +sin(theta_b-theta_a); to node b is -sin(...)
    for e, (a, b) in enumerate(pairs):
        col = n + e
        s = np.sin(theta[:, b] - theta[:, a]).astype(np.float32)
        rows_a = t_idx * n + a
        rows_b = t_idx * n + b
        Theta_flat[rows_a, col] = s
        Theta_flat[rows_b, col] = -s

    # triangle-incidence columns: for each triple (i<j<k) and each center c in {i,j,k}
    # column only affects the center node row
    base = n + mE
    col = base
    for (i, j, k) in triples:
        # center k: sin(i + j - 2k) contributes only to node k
        s_k = np.sin(theta[:, i] + theta[:, j] - 2.0 * theta[:, k]).astype(np.float32)
        Theta_flat[t_idx * n + k, col] = s_k
        col += 1

        # center j: sin(i + k - 2j) contributes only to node j
        s_j = np.sin(theta[:, i] + theta[:, k] - 2.0 * theta[:, j]).astype(np.float32)
        Theta_flat[t_idx * n + j, col] = s_j
        col += 1

        # center i: sin(j + k - 2i) contributes only to node i
        s_i = np.sin(theta[:, j] + theta[:, k] - 2.0 * theta[:, i]).astype(np.float32)
        Theta_flat[t_idx * n + i, col] = s_i
        col += 1

    assert col == p
    return Theta_flat, Y_flat, pairs, triples

# ---------------- Build SOC constraints: triangle-incidence -> parent edges ----------------
def build_soc_constraints(n, pairs, triples):
    """
    Returns:
      - edges_soc: list of (child_row, parent_row) indices in coefficient vector Xi (g rows)
      - maps for readout
    Coefficient rows are aligned with columns of Theta_flat:
      0..n-1          : omega
      n..n+mE-1       : edges
      n+mE..end       : triangle-incidences (3 per triple in order: center k, center j, center i)
    """
    pair_to_row = {}
    for e, (a, b) in enumerate(pairs):
        pair_to_row[(a, b)] = n + e  # edge row in Xi

    tri_to_rows = {}  # frozenset({i,j,k}) -> dict(center -> row)
    edges_soc = []

    base = n + len(pairs)
    col = base
    for (i, j, k) in triples:
        tri_key = frozenset((i, j, k))
        tri_to_rows.setdefault(tri_key, {})

        # order: center k, center j, center i
        tri_to_rows[tri_key][k] = col; col += 1
        tri_to_rows[tri_key][j] = col; col += 1
        tri_to_rows[tri_key][i] = col; col += 1

    # SOC edges from each triangle-incidence to each of its 3 parent edges
    for tri_key, center_map in tri_to_rows.items():
        a, b, c = sorted(list(tri_key))
        parent_edges = [tuple(sorted((a, b))), tuple(sorted((a, c))), tuple(sorted((b, c)))]
        parent_rows = []
        for pe in parent_edges:
            if pe in pair_to_row:
                parent_rows.append(pair_to_row[pe])

        for center, child_row in center_map.items():
            for pr in parent_rows:
                edges_soc.append((child_row, pr))

    return edges_soc, pair_to_row, tri_to_rows

# ---------------- Scores + F1 ----------------
def f1_from_threshold(pred_set, gt_set):
    TP = len(pred_set & gt_set)
    FP = len(pred_set - gt_set)
    FN = len(gt_set - pred_set)
    P = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    R = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    F1 = 2 * P * R / (P + R) if (P + R) > 0 else 0.0
    return P, R, F1, TP, FP, FN

def sweep_thresholds(vals_dict, gt_set, qs):
    """
    vals_dict: mapping key -> score (nonnegative)
    Returns best F1 over quantile thresholds of the score distribution.
    """
    if not vals_dict:
        return 0.0, None

    vals = np.array(list(vals_dict.values()), dtype=float)
    best = (0.0, None)
    for q in qs:
        tau = float(np.quantile(vals, q))
        pred = {k for k, v in vals_dict.items() if v >= tau}
        _, _, f1, *_ = f1_from_threshold(pred, gt_set)
        if f1 > best[0]:
            best = (f1, (q, tau))
    return best

# ---------------- Main pipeline for one (lambda_scale, rho_hier) ----------------
def run_one(theta, dtheta, n, scale, rho_hier):
    # build design
    Theta_flat, Y_flat, pairs, triples = build_design(theta, dtheta, n, MAX_ROWS_TIME)

    # column scaling (NO mean-centering; keep omega columns as-is)
    col_std = Theta_flat.std(axis=0).astype(np.float32)
    col_scale = np.ones_like(col_std)
    # omega columns are constant ones; std=0 -> keep scale 1
    nonconst = col_std > 1e-8
    col_scale[nonconst] = col_std[nonconst]
    Theta_s = (Theta_flat / col_scale[None, :]).astype(np.float32)

    # scale target lightly for numerical stability
    y_scale = robust_sigma(Y_flat) + 1e-12
    Y_s = (Y_flat / y_scale).astype(np.float32)

    # move to GPU if available
    if GPU_AVAILABLE:
        Theta_s = cp.asarray(Theta_s)
        Y_s = cp.asarray(Y_s)

    # precompute Gram
    Nsamples = Theta_s.shape[0]
    G = (Theta_s.T @ Theta_s) / float(Nsamples)
    # tiny ridge for stability
    G = G + (5e-4 * xp.eye(G.shape[0], dtype=G.dtype))
    B = (Theta_s.T @ Y_s) / float(Nsamples)

    # SOC constraints
    edges_soc, pair_to_row, tri_to_rows = build_soc_constraints(n, pairs, triples)

    # weights w for group lasso (row-wise). omega rows unpenalized -> w=0
    g = int(G.shape[0])
    w = xp.ones(g, dtype=G.dtype)
    w[:n] = 0.0

    # lambda heuristic
    # Using robust sigma of Y_s means sigma ~1; keep standard scaling with sqrt(2 log p / N)
    p = g
    sigma_hat = 1.0
    lam = float(scale) * sigma_hat * math.sqrt(2.0 * math.log(max(3, p)) / max(2.0, Nsamples))

    # solve
    Xi = solve_admm_gl_soc(
        G, B, edges_soc,
        lamb=lam, rho_hier=float(rho_hier), w=w,
        max_iters=MAX_ITERS, rho_admm=ADMM_RHO, overrelax=ADMM_OVERRELAX,
        adaptive_rho=True, log_every=0,
        proj_every=1, proj_iters=60, proj_tol=1e-9,
    )

    # bring back to CPU
    Xi_cpu = cp.asnumpy(Xi).reshape(-1) if GPU_AVAILABLE else np.asarray(Xi).reshape(-1)

    # unscale coefficients back to original units:
    # Y_flat ≈ Theta_flat * beta  ->  (Y_s*y_scale) ≈ (Theta_s*col_scale) * beta
    # Theta_s = Theta_flat / col_scale, Y_s = Y_flat / y_scale
    # so beta_original = (y_scale / col_scale) * beta_solved
    beta = (y_scale / col_scale) * Xi_cpu

    # edge scores
    edge_scores = {}
    for e, (a, b) in enumerate(pairs):
        row = n + e
        edge_scores[frozenset((a, b))] = float(abs(beta[row]))

    # triangle scores: for each triangle, take max over its 3 center-incidences
    # triangle scores: for each triangle, compute max/mean/geom over its 3 center-incidences
    tri_scores_max = {}
    tri_scores_mean = {}
    tri_scores_geom = {}

    for tri_key, center_map in tri_to_rows.items():

      mags = [float(abs(beta[row])) for row in center_map.values()]
      a, b, c = mags  # exactly 3

      tri_scores_max[tri_key]  = max(a, b, c)
      tri_scores_mean[tri_key] = (a + b + c) / 3.0
      tri_scores_geom[tri_key] = (a * b * c) ** (1.0 / 3.0)


    # compute best F1 by sweeping quantiles (avoid picking an arbitrary threshold)
    edge_best_f1, edge_best_info = sweep_thresholds(edge_scores, GT_EDGES_SET, qs=np.linspace(0.30, 0.99, 25))
    tri_best_f1_max,  tri_best_info_max  = sweep_thresholds(tri_scores_max,  GT_TRIS_SET, qs=np.linspace(0.30, 0.99, 25))
    tri_best_f1_mean, tri_best_info_mean = sweep_thresholds(tri_scores_mean, GT_TRIS_SET, qs=np.linspace(0.30, 0.99, 25))
    tri_best_f1_geom, tri_best_info_geom = sweep_thresholds(tri_scores_geom, GT_TRIS_SET, qs=np.linspace(0.30, 0.99, 25))


    # also return estimated omega for sanity
    omega_hat = beta[:n]

    return {
    "scale": float(scale),
    "rho": float(rho_hier),
    "lambda": float(lam),
    "edge_F1": float(edge_best_f1),

    "tri_F1_max":  float(tri_best_f1_max),
    "tri_F1_mean": float(tri_best_f1_mean),
    "tri_F1_geom": float(tri_best_f1_geom),

    "edge_best": edge_best_info,
    "tri_best_max":  tri_best_info_max,
    "tri_best_mean": tri_best_info_mean,
    "tri_best_geom": tri_best_info_geom,

    "omega_hat": omega_hat,
    "beta": beta,
}


# ---------------- Load data ----------------
def load_csv_matrix(path: str) -> np.ndarray:
    arr = pd.read_csv(path, header=None).values.astype(np.float64)
    return arr

def ensure_T_by_n(mat: np.ndarray, n_expected: int = None) -> np.ndarray:
    """
    Your generator saves (steps, n). Some scripts transpose.
    We'll detect the intended shape.
    """
    if n_expected is not None:
        if mat.shape[1] == n_expected:
            return mat
        if mat.shape[0] == n_expected:
            return mat.T
    # fallback: assume time dimension is the larger one
    return mat if mat.shape[0] >= mat.shape[1] else mat.T

def main():
    if not os.path.exists(THETA_CSV):
        raise FileNotFoundError(f"Missing {THETA_CSV}")
    if not os.path.exists(DTHETA_CSV):
        raise FileNotFoundError(f"Missing {DTHETA_CSV}")

    theta_raw = load_csv_matrix(THETA_CSV)
    dtheta_raw = load_csv_matrix(DTHETA_CSV)

    theta = ensure_T_by_n(theta_raw, n_expected=N_NODES)
    dtheta = ensure_T_by_n(dtheta_raw, n_expected=N_NODES)

    T, n = theta.shape
    print(f"Loaded: theta shape={theta.shape}, dtheta shape={dtheta.shape}")

    if n != N_NODES:
        print(f"⚠️ Warning: n={n} differs from N_NODES={N_NODES}. Using n={n} from data.")
    n = int(n)

    # Your simulator does not wrap theta; no unwrap needed.
    # If you ever wrap to [-pi,pi], uncomment:
    # theta = np.unwrap(theta, axis=0)

    all_results = []
    print("\nScale    Rho    EdgeF1  TriMax  TriMean TriGeom")
    print("------------------------------------------------")

    print("-" * 32)

    for scale in LAMBDA_SCALES:
        for rho in RHO_SCALES:
            res = run_one(theta, dtheta, n, scale=scale, rho_hier=rho)
            all_results.append({

                "scale": res["scale"],
                "rho": res["rho"],
                "lambda": res["lambda"],
                "edge_F1": res["edge_F1"],

                "tri_F1_max":  res["tri_F1_max"],
                "tri_F1_mean": res["tri_F1_mean"],
                "tri_F1_geom": res["tri_F1_geom"],

                "edge_best_q_tau": str(res["edge_best"]),
                "tri_best_max_q_tau":  str(res["tri_best_max"]),
                "tri_best_mean_q_tau": str(res["tri_best_mean"]),
                "tri_best_geom_q_tau": str(res["tri_best_geom"]),
            })


            print(f"{res['scale']:<7.3f} {res['rho']:<6.2f} "
            f"{res['edge_F1']:<7.3f} {res['tri_F1_max']:<7.3f} "
            f"{res['tri_F1_mean']:<7.3f} {res['tri_F1_geom']:<7.3f}")

    df = pd.DataFrame(all_results)
    out_csv = os.path.join(OUTDIR, "grid_search_correct_incidence.csv")
    df.to_csv(out_csv, index=False)
    print(f"\n✅ Saved: {out_csv}")

    # show best configs
    best_edge = df.sort_values("edge_F1", ascending=False).head(5)
    best_tri  = df.sort_values("tri_F1_max", ascending=False).head(5)
    best_both = df.assign(sumF1=df.edge_F1 + df.tri_F1_max).sort_values("sumF1", ascending=False).head(5)

    print("\nTop-5 by Edge F1:\n",
      best_edge[["scale","rho","lambda","edge_F1","tri_F1_max","tri_F1_mean","tri_F1_geom"]].to_string(index=False))

    print("\nTop-5 by Tri F1 (MAX):\n",
      best_tri[["scale","rho","lambda","edge_F1","tri_F1_max","tri_F1_mean","tri_F1_geom"]].to_string(index=False))

    print("\nTop-5 by Edge + TriMax:\n",
      best_both[["scale","rho","lambda","edge_F1","tri_F1_max","tri_F1_mean","tri_F1_geom","sumF1"]].to_string(index=False))

    print("\nTop-5 by Edge F1:\n", best_edge[["scale","rho","lambda","edge_F1","tri_F1"]].to_string(index=False))
    print("\nTop-5 by Tri F1:\n",  best_tri[["scale","rho","lambda","edge_F1","tri_F1"]].to_string(index=False))
    print("\nTop-5 by Edge+Tri F1:\n", best_both[["scale","rho","lambda","edge_F1","tri_F1","sumF1"]].to_string(index=False))

if __name__ == "__main__":
    main()
